# Tutorial 11 — RAG Pipeline over Scientific Literature
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Retrieval-Augmented Generation (RAG) combines a vector database of documents with an LLM — the LLM answers questions grounded in retrieved context. This is transformative for scientific Q&A: instead of hallucinating, the model cites real papers.

In [ ]:
!pip install langchain langchain-community langchain-anthropic -q
!pip install chromadb sentence-transformers -q
!pip install requests -q

## Step 1 — Fetch PubMed abstracts

In [ ]:
import requests, xml.etree.ElementTree as ET
from langchain.schema import Document

def fetch_pubmed_abstracts(query: str, max_results: int = 20) -> list[Document]:
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    # Search
    search = requests.get(f"{base}esearch.fcgi",
                          params={"db":"pubmed","term":query,
                                  "retmax":max_results,"retmode":"json"})
    pmids = search.json()["esearchresult"]["idlist"]
    if not pmids:
        return []
    # Fetch
    fetch = requests.get(f"{base}efetch.fcgi",
                         params={"db":"pubmed","id":",".join(pmids),
                                 "rettype":"abstract","retmode":"xml"})
    root = ET.fromstring(fetch.text)
    docs = []
    for art in root.iter("PubmedArticle"):
        title    = "".join(t.itertext() for t in art.iter("ArticleTitle"))
        abstract = "".join(t.itertext() for t in art.iter("AbstractText"))
        pmid     = art.findtext(".//PMID","")
        year     = art.findtext(".//PubDate/Year","?")
        if abstract.strip():
            docs.append(Document(
                page_content=f"Title: {title}\n\nAbstract: {abstract}",
                metadata={"pmid": pmid, "title": title, "year": year}
            ))
    return docs

print("Fetching PubMed abstracts on SILCS drug discovery...")
docs = fetch_pubmed_abstracts("SILCS protein ligand binding drug discovery", max_results=15)
print(f"Fetched {len(docs)} abstracts")
for d in docs[:3]:
    print(f"  [{d.metadata['year']}] PMID {d.metadata['pmid']}: {d.metadata['title'][:70]}...")

## Step 2 — Embed and store in ChromaDB

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
if docs:
    vectordb = Chroma.from_documents(docs, embedder, persist_directory="./pubmed_chroma")
    print(f"Indexed {len(docs)} documents into ChromaDB")

    # Test retrieval
    query = "protein-ligand binding affinity prediction"
    retrieved = vectordb.similarity_search(query, k=3)
    print(f"\nTop 3 results for: '{query}'")
    for r in retrieved:
        print(f"  [{r.metadata['year']}] {r.metadata['title'][:60]}...")
else:
    print("No docs fetched — check your internet connection")

## Step 3 — RAG Q&A chain

In [ ]:
# NOTE: Set your API key as an environment variable:
# import os; os.environ["ANTHROPIC_API_KEY"] = "your-key-here"
# Then run this cell.

RAG_CODE = '''
from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA

llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)
qa  = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
)

questions = [
    "What is the SILCS method and how does it compute binding affinity?",
    "How does SILCS compare to FEP methods for protein-ligand binding?",
    "What is the application of SILCS to hERG cardiotoxicity prediction?",
]

for q in questions:
    print(f"Q: {q}")
    result = qa.invoke(q)
    print(f"A: {result['result'][:300]}...")
    print(f"Sources: {[d.metadata['pmid'] for d in result['source_documents']]}")
    print()
'''
print("RAG chain code ready. Set ANTHROPIC_API_KEY and run:")
print(RAG_CODE)

## Key takeaways
- RAG = Retrieval + Generation: ground LLM answers in real documents
- ChromaDB + sentence-transformers enables local, free embedding and retrieval
- The Colab badge URL pattern lets anyone run this notebook without installing anything
- For production: use larger models, add metadata filters, implement re-ranking